# Time travel × external mappers — cache build (human)

This stage builds *reusable caches* for a marketing-critical question:

**How well can point-in-time external identifier mappers cope with historical Ensembl IDs without a time-travel layer?**

IDTrack’s value proposition is that identifiers live in a coordinate system
(`namespace × Ensembl time × assembly`). External mappers are valuable services,
but most are *not* time-travel systems and (by design) do not make the time boundary
an explicit parameter.

This cache notebook produces two comparable regimes for each target namespace (HGNC, UniProt):

- **Naive external mapping**: map *old* Ensembl gene IDs directly to a target namespace (no time travel).
- **Time-travel assisted mapping**: first map old IDs → `to_release` Ensembl IDs with IDTrack, then map those to the target namespace with each external mapper.

The downstream analysis notebook (`01_...`) converts these caches into manuscript-ready multi-panel figures and tables.

## Design goals (why this is built this way)

- **Frictionless reruns**: caches are written per `(from_release, bootstrap)` bundle; reruns resume where they left off.
- **No hidden switches**: the notebook is cache-first; if caches exist, it won’t recompute.
- **Manuscript consistency**: uses the shared rcParams and output conventions from `experiments_utils.py`.

## Expected results (what to look for)

- For older `from_release`, naive external mapping should lose coverage (more `1:0`) because many historical IDs are no longer valid inputs at the target time boundary.
- After time travel, external tools should recover coverage (because inputs are now valid at `to_release`), but may still disagree with each other and/or surface ambiguity (`1:n`).
- IDTrack provides an end-to-end mapping with explicit ambiguity; it should remain stable across the time axis (given a fixed `snapshot_release`).


In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

import os
import sys

# Add experiments/src to sys.path (layout-aware; works on Slurm and locally)
REPO_ROOT = Path(os.environ.get('REPO_ROOT', Path.cwd())).expanduser().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (
    (REPO_ROOT / 'idtrack').is_dir()
    and ((REPO_ROOT / 'reproducibility').is_dir() or (REPO_ROOT / 'idtrack' / 'reproducibility').is_dir())
):
    REPO_ROOT = REPO_ROOT.parent

REPRO_ROOT = REPO_ROOT / 'reproducibility' if (REPO_ROOT / 'reproducibility').is_dir() else REPO_ROOT / 'idtrack' / 'reproducibility'
EXPERIMENTS_SRC = REPRO_ROOT / 'experiments' / 'src'
if str(EXPERIMENTS_SRC) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    notebook_context,
    stable_hash,
    write_json,
    write_pickle,
    read_json,
    read_pickle,
)

from external_mapper_eval import ExternalRun, run_external_methods  # noqa: E402
from idtrack_results import summarize_matchings  # noqa: E402

ctx = notebook_context('time_travel_vs_external_mappers', start=REPO_ROOT)

IDTRACK_LOCAL_REPO = ctx.idtrack_local_repo
CACHE_DIR = ctx.experiment_cache

print('REPO_ROOT:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Configuration --------------------

ORGANISM_ALIAS = 'human'

# Snapshot boundary: must be <= what you have cached locally in IDTRACK_LOCAL_REPO.
SNAPSHOT_RELEASE = 114

# The target time boundary we want to express results in.
TO_RELEASE = 114

# External namespaces we care about for the manuscript.
TARGET_DATABASES = ['HGNC Symbol', 'UniProtKB/Swiss-Prot']

# External tools to compare (uses idtrack._external_mappers).
EXTERNAL_METHODS = ['pybiomart', 'mygene', 'gprofiler', 'gget']

# Grid density controls cost.
# - 'paper': coarse but fast; good first run
# - 'dense': more evidence without exploding runtime
# - 'full': every release (can be heavy; use Slurm)
RELEASE_GRID = 'dense'
if RELEASE_GRID == 'paper':
    FROM_RELEASES = list(range(75, 115, 5))
elif RELEASE_GRID == 'full':
    FROM_RELEASES = list(range(75, 115, 1))
else:  # dense
    FROM_RELEASES = list(range(75, 115, 2))

# Sampling and uncertainty
N_POOL_PER_FROM = 5000
N_IDS_PER_FROM = 1000
N_BOOTSTRAPS = 6
RANDOM_SEED = 0

# Mapping semantics: expose ambiguity (1→n) rather than collapsing it.
STRATEGY = 'all'

# Recompute bundles even if cache exists
FORCE = False

PARAMS = {
    'organism_alias': ORGANISM_ALIAS,
    'snapshot_release': int(SNAPSHOT_RELEASE),
    'to_release': int(TO_RELEASE),
    'from_releases': [int(x) for x in FROM_RELEASES],
    'target_databases': list(TARGET_DATABASES),
    'external_methods': list(EXTERNAL_METHODS),
    'release_grid': str(RELEASE_GRID),
    'n_pool_per_from': int(N_POOL_PER_FROM),
    'n_ids_per_from': int(N_IDS_PER_FROM),
    'n_bootstraps': int(N_BOOTSTRAPS),
    'random_seed': int(RANDOM_SEED),
    'strategy': str(STRATEGY),
}

FP = stable_hash(json.dumps(PARAMS, sort_keys=True), n=12)
PARAMS_JSON = CACHE_DIR / f'time_travel_vs_external_mappers_params_{FP}.json'
POOLS_JSON = CACHE_DIR / f'time_travel_vs_external_mappers_pools_{FP}.json'

print('Fingerprint:', FP)
print('PARAMS_JSON:', PARAMS_JSON)
print('POOLS_JSON:', POOLS_JSON)
print('FROM_RELEASES:', FROM_RELEASES[:10], '...' if len(FROM_RELEASES) > 10 else '')
print('TARGET_DATABASES:', TARGET_DATABASES)
print('EXTERNAL_METHODS:', EXTERNAL_METHODS)


In [ ]:
# -------------------- Pools (reuse if possible; otherwise build) --------------------

def _coerce_pools(raw: dict) -> dict[int, list[str]]:
    pools = raw.get('pools', raw)
    out: dict[int, list[str]] = {}
    for k, v in (pools or {}).items():
        try:
            kk = int(k)
        except Exception:
            continue
        out[kk] = [str(x) for x in (v or [])]
    return out

def _try_reuse_time_travel_matrix_pools() -> dict[int, list[str]] | None:
    other = Path(IDTRACK_LOCAL_REPO) / 'experiments' / 'time_travel_matrix'
    candidates = list(other.glob('time_travel_matrix_pools_*.json'))
    if not candidates:
        return None
    p = max(candidates, key=lambda x: x.stat().st_mtime)
    try:
        data = read_json(p)
        pools = _coerce_pools(data)
    except Exception as e:  # noqa: BLE001
        print('Could not reuse pools from', p, ':', e)
        return None
    if not pools:
        return None
    missing = [r for r in FROM_RELEASES if int(r) not in pools]
    if missing:
        print('Pools reuse skipped: missing releases in reused pools:', missing[:8], '...' if len(missing) > 8 else '')
        return None
    return pools

pools: dict[int, list[str]] | None = None
if POOLS_JSON.exists() and not FORCE:
    pools = _coerce_pools(read_json(POOLS_JSON))
    print('Loaded pools from:', POOLS_JSON)
else:
    pools = _try_reuse_time_travel_matrix_pools()
    if pools is not None:
        print('Reusing pools from experiment_time_travel_matrix (latest cache).')
        write_json({'pools': pools, 'source': 'reused_time_travel_matrix'}, POOLS_JSON)
    else:
        print('No reusable pools found; pools will be built from the graph during cache build.')

pools = pools or {}
print('Pool releases available:', len(pools))


In [ ]:
# -------------------- Build per-(from_release, bootstrap) bundles --------------------

# Determine which bundles are missing.
tasks = []
for b in range(int(N_BOOTSTRAPS)):
    for fr in FROM_RELEASES:
        fr = int(fr)
        out = CACHE_DIR / f'time_travel_vs_external_mappers_bundle_{FP}_from{fr}_b{b}.pickle'
        if FORCE or (not out.exists()):
            tasks.append((b, fr, out))

if not tasks:
    print('All bundles already exist. Nothing to do.')
else:
    import idtrack
    from idtrack._the_graph import TheGraph

    write_json(PARAMS, PARAMS_JSON)

    api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
    api.configure_logger()
    organism, latest = api.resolve_organism(ORGANISM_ALIAS)
    snapshot = int(SNAPSHOT_RELEASE)
    if snapshot > int(latest):
        raise ValueError(f'snapshot_release={snapshot} exceeds latest={latest} for {ORGANISM_ALIAS}')
    if int(TO_RELEASE) > snapshot:
        raise ValueError(f'to_release={TO_RELEASE} exceeds snapshot_release={snapshot}')

    print(f'Building/loading graph: {organism} snapshot_release={snapshot}')
    api.build_graph(organism_name=organism, snapshot_release=snapshot, calculate_caches=True)
    g = api.track.graph

    # Build pools if missing (graph scan; cache-first).
    if (not pools) or any(int(r) not in pools for r in FROM_RELEASES):
        rng_pool = np.random.default_rng(int(RANDOM_SEED))
        pools = {int(r): [] for r in FROM_RELEASES}
        seen = {int(r): 0 for r in FROM_RELEASES}

        def _reservoir_update(r: int, item: str) -> None:
            seen[r] += 1
            if len(pools[r]) < int(N_POOL_PER_FROM):
                pools[r].append(item)
                return
            j = int(rng_pool.integers(0, seen[r]))
            if j < int(N_POOL_PER_FROM):
                pools[r][j] = item

        t0_pool = time.perf_counter()
        for node in g.nodes:
            if not isinstance(node, str) or not node.startswith('ENSG'):
                continue
            try:
                ranges = g.get_active_ranges_of_id[node]
            except Exception:
                continue
            for r in FROM_RELEASES:
                rr = int(r)
                if TheGraph.is_point_in_range(ranges, rr):
                    _reservoir_update(rr, node)

        dt_pool = time.perf_counter() - t0_pool
        print(f'Built pools in {dt_pool:.1f}s')
        write_json({'pools': pools, 'seen': seen, 'source': 'built_local'}, POOLS_JSON)

    print('Planned bundles:', len(tasks))

    for b, fr, out in tasks:
        rng = np.random.default_rng(int(RANDOM_SEED) + 10_000 + int(b) * 97 + int(fr))
        pool = pools.get(int(fr), [])
        if not pool:
            print(f'SKIP from_release={fr}: empty pool')
            continue

        ids_from = rng.choice(pool, size=min(int(N_IDS_PER_FROM), len(pool)), replace=False).tolist()
        ids_from = [str(x) for x in ids_from]

        # 1) IDTrack backbone time travel (old ENSG -> ENSG at to_release)
        t0 = time.perf_counter()
        backbone = api.convert_identifier_multiple(
            ids_from.copy(),
            from_release=int(fr),
            to_release=int(TO_RELEASE),
            final_database=None,
            strategy=STRATEGY,
            verbose=False,
        )
        dt_backbone = time.perf_counter() - t0

        # Extract the 1→1 subset (unique, valid ENSG at to_release).
        to_ids = []
        n_backbone_1to1 = 0
        for item in backbone:
            if item.get('no_corresponding') or item.get('no_conversion'):
                continue
            targets = item.get('target_id') or []
            uniq = []
            seen = set()
            for t in targets:
                if t is None:
                    continue
                s = str(t).strip()
                if not s or s.lower() in {'nan', 'none', 'null'}:
                    continue
                if s not in seen:
                    seen.add(s)
                    uniq.append(s)
            if len(uniq) == 1:
                n_backbone_1to1 += 1
                to_ids.append(uniq[0])

        to_ids = sorted(set(to_ids))

        # 2) IDTrack end-to-end mapping (old ENSG -> target namespace at to_release)
        idt_old: dict[str, list[dict]] = {}
        idt_to: dict[str, list[dict]] = {}
        timings: dict[str, float] = {'idtrack_backbone_seconds': float(dt_backbone)}

        for target_db in TARGET_DATABASES:
            t0 = time.perf_counter()
            idt_old[target_db] = api.convert_identifier_multiple(
                ids_from.copy(),
                from_release=int(fr),
                to_release=int(TO_RELEASE),
                final_database=target_db,
                strategy=STRATEGY,
                verbose=False,
            )
            timings[f'idtrack_old_to_{target_db}_seconds'] = float(time.perf_counter() - t0)

            t0 = time.perf_counter()
            idt_to[target_db] = api.convert_identifier_multiple(
                to_ids.copy(),
                from_release=int(TO_RELEASE),
                to_release=int(TO_RELEASE),
                final_database=target_db,
                strategy=STRATEGY,
                verbose=False,
            ) if to_ids else []
            timings[f'idtrack_to_to_{target_db}_seconds'] = float(time.perf_counter() - t0)

        # 3) External mappers
        external = {'naive': {}, 'time_travel_assisted': {}}
        external_errors: list[dict] = []

        for target_db in TARGET_DATABASES:
            # Naive: old IDs directly
            run_naive = ExternalRun(
                ids=ids_from,
                input_db='ensembl_gene',
                output_db=target_db,
                species=ORGANISM_ALIAS,
                methods=tuple(EXTERNAL_METHODS),
                pybiomart_release=int(TO_RELEASE),
                chunk_size=200,
                pause=0.1,
                max_retries=3,
                strip_versions=True,
                verbose=2,
                suppress_method_verbosity=True,
            )
            t0 = time.perf_counter()
            res_naive, err_naive = run_external_methods(run_naive)
            timings[f'external_naive_{target_db}_seconds'] = float(time.perf_counter() - t0)
            external['naive'][target_db] = {k: v for k, v in res_naive.items()}
            external_errors.extend([{**e, 'scenario': 'naive', 'target_db': target_db} for e in err_naive])

            # Time-travel assisted: to_release IDs
            run_tt = ExternalRun(
                ids=to_ids,
                input_db='ensembl_gene',
                output_db=target_db,
                species=ORGANISM_ALIAS,
                methods=tuple(EXTERNAL_METHODS),
                pybiomart_release=int(TO_RELEASE),
                chunk_size=200,
                pause=0.1,
                max_retries=3,
                strip_versions=True,
                verbose=2,
                suppress_method_verbosity=True,
            )
            t0 = time.perf_counter()
            res_tt, err_tt = run_external_methods(run_tt)
            timings[f'external_time_travel_{target_db}_seconds'] = float(time.perf_counter() - t0)
            external['time_travel_assisted'][target_db] = {k: v for k, v in res_tt.items()}
            external_errors.extend([{**e, 'scenario': 'time_travel_assisted', 'target_db': target_db} for e in err_tt])

        # Quick sanity summaries for the cache file (useful when browsing).
        idt_old_summary = {t: summarize_matchings(m) for t, m in idt_old.items()}
        idt_to_summary = {t: summarize_matchings(m) for t, m in idt_to.items()}

        bundle = {
            'fingerprint': FP,
            'params': PARAMS,
            'from_release': int(fr),
            'to_release': int(TO_RELEASE),
            'bootstrap': int(b),
            'ids_from': ids_from,
            'backbone_matchings': backbone,
            'n_backbone_1to1': int(n_backbone_1to1),
            'to_ids_1to1_unique': to_ids,
            'idtrack_old_to_target_matchings': idt_old,
            'idtrack_to_to_target_matchings': idt_to,
            'idtrack_old_to_target_summary': idt_old_summary,
            'idtrack_to_to_target_summary': idt_to_summary,
            'external_results': external,
            'external_errors': external_errors,
            'timings': timings,
        }

        write_pickle(bundle, out)
        print(f'Wrote bundle: from_release={fr} bootstrap={b} -> {out.name} (n_ids={len(ids_from)}, n_to_ids={len(to_ids)})')


## Post-run report (strongly recommended)

This summary is cheap (just reads the written bundle pickles) but extremely helpful when running on Slurm:

- Confirms how many bundles were produced for this fingerprint
- Surfaces external-mapper dependency / API failures early
- Creates a compact CSV you can quickly grep/plot without opening notebooks


In [ ]:
from experiments_utils import atomic_write_dataframe_csv  # noqa: E402

bundle_paths = sorted(CACHE_DIR.glob(f'time_travel_vs_external_mappers_bundle_{FP}_from*_b*.pickle'))
print('Bundles on disk:', len(bundle_paths))
if not bundle_paths:
    raise FileNotFoundError('No bundles found for this fingerprint. Did the build loop run?')

rows = []
err_rows = []
for p in bundle_paths:
    b = read_pickle(p)
    timings = b.get('timings', {}) or {}
    errs = b.get('external_errors', []) or []
    rows.append(
        {
            'fingerprint': b.get('fingerprint'),
            'from_release': int(b.get('from_release')),
            'to_release': int(b.get('to_release')),
            'bootstrap': int(b.get('bootstrap')),
            'n_ids_from': int(len(b.get('ids_from', []) or [])),
            'n_backbone_1to1': int(b.get('n_backbone_1to1', 0) or 0),
            'n_to_ids_1to1_unique': int(len(b.get('to_ids_1to1_unique', []) or [])),
            'n_external_errors': int(len(errs)),
            **{f't_{k}': float(v) for k, v in timings.items() if v is not None},
            'bundle_file': p.name,
        }
    )
    for e in errs:
        err_rows.append({'bundle_file': p.name, **(e or {})})

report = pd.DataFrame(rows).sort_values(['from_release', 'bootstrap']).reset_index(drop=True)
errors = pd.DataFrame(err_rows)

out_report = CACHE_DIR / f'time_travel_vs_external_mappers_build_report_{FP}.csv'
atomic_write_dataframe_csv(report, out_report, index=False)
print('Wrote:', out_report)
display(report.head(12))

if not errors.empty:
    out_err = CACHE_DIR / f'time_travel_vs_external_mappers_external_errors_{FP}.csv'
    atomic_write_dataframe_csv(errors, out_err, index=False)
    print('Wrote:', out_err)
    display(errors.head(15))
